# Data Validation

Upload a CSV file and validate it against the OpenGLASSLab data schema for glazing data, harvest data, or test data.


<div style="border-left: 5px solid #0074D9; padding: 10px;">
    <strong>Note:</strong> You can use the 🚀 <strong>`Live Code`</strong> button in the top right to activate the interactive features and use Python interactively! <br><br>
                           Once the "Live Code" is enabled it is advised to <strong>"Run All"</strong> cells first to load all the necessary packages and functions. <br><br>
                           Afterwards, upload a CSV file below, select the matching data type from the dropdown, and click <strong>"Validate"</strong> to run the corresponding validation script.
</div>


In [ ]:
import sys
import tempfile
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, HTML

# "emscripten" is the sys.platform reported inside a Pyodide (browser/Live Code) kernel.
IS_PYODIDE = sys.platform == "emscripten"

if IS_PYODIDE:
    # Live Code (thebe-lite) runs in an in-browser Pyodide sandbox with no access
    # to this repo's filesystem, so the validation scripts are fetched directly
    # from GitHub and loaded as modules 
    import types

    import micropip
    from pyodide.http import pyfetch

    await micropip.install(["pandas", "pydantic"])

    _RAW_BASE = "https://raw.githubusercontent.com/TUDelft-books/open-GLASSlab/data-validation/book/scripts/"
    _MODULE_NAMES = ["validate_glazing_data", "validate_harvest_data", "validate_test_data"]

    for _name in _MODULE_NAMES:
        _response = await pyfetch(f"{_RAW_BASE}{_name}.py")
        _source = await _response.string()
        _module = types.ModuleType(_name)
        exec(compile(_source, f"{_name}.py", "exec"), _module.__dict__)
        sys.modules[_name] = _module

    import validate_glazing_data
    import validate_harvest_data
    import validate_test_data
else:
    # Running locally (VS Code / Jupyter): import directly from the repo's
    # `scripts` folder, one level up from this notebook (book/pages/ -> book/scripts/).
    _SCRIPTS_DIR = (Path.cwd() / ".." / "scripts").resolve()

    if not _SCRIPTS_DIR.is_dir():
        raise FileNotFoundError(f"Could not find the 'scripts' folder at: {_SCRIPTS_DIR}")

    if str(_SCRIPTS_DIR) not in sys.path:
        sys.path.insert(0, str(_SCRIPTS_DIR))

    import validate_glazing_data
    import validate_harvest_data
    import validate_test_data

# Maps each dropdown option to the matching validation script's `validate_file` function
VALIDATORS = {
    "Glazing data": validate_glazing_data.validate_file,
    "Harvesting data": validate_harvest_data.validate_file,
    "Test data": validate_test_data.validate_file,
}


In [ ]:
# Create an HTML widget for instructions
instructions = widgets.HTML(
    value="""
    <p style="font-size: 16px">Upload a CSV file, select the data type it contains, then click <b>Validate</b>.</p>
    <b style="font-size: 16px">Data protection declaration: The uploaded file is processed locally and is not transmitted over the network or saved permanently.</b>
    """
)

# Widget to upload a CSV file
upload_widget = widgets.FileUpload(
    description="Upload CSV",
    accept=".csv",
    multiple=False,
    layout=widgets.Layout(width="250px"),
)

# Dropdown to select which validation script should be run
data_type_dropdown = widgets.Dropdown(
    options=["Glazing data", "Harvesting data", "Test data"],
    description="Data type:",
    style={"description_width": "auto"},
    layout=widgets.Layout(width="300px"),
)

# Button to trigger validation
validate_button = widgets.Button(
    description="Validate",
    button_style="success",
    icon="check",
)

# Output container for the validation report
output = widgets.Output()


def _get_uploaded_file():
    """Return (filename, content_bytes) for the uploaded file, or (None, None)."""
    value = upload_widget.value
    if not value:
        return None, None
    if isinstance(value, dict):
        # ipywidgets 7.x API: {filename: {"content": ..., "metadata": ...}}
        filename, info = next(iter(value.items()))
    else:
        # ipywidgets 8.x API: tuple of dicts with "name" and "content"
        info = value[0]
        filename = info["name"]
    return filename, bytes(info["content"])


def on_validate_clicked(_):
    output.clear_output()
    with output:
        filename, content = _get_uploaded_file()
        if filename is None:
            print("Please upload a CSV file before clicking Validate.")
            return

        validator = VALIDATORS[data_type_dropdown.value]

        with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
            tmp.write(content)
            tmp_path = Path(tmp.name)

        print(f"Uploaded file: {filename}")
        print(f"Validating as: {data_type_dropdown.value}\n")

        try:
            passed = validator(tmp_path)
        finally:
            tmp_path.unlink(missing_ok=True)

        if passed:
            display(HTML('<p style="color: green; font-weight: bold;">✅ Validation passed.</p>'))
        else:
            display(HTML('<p style="color: red; font-weight: bold;">❌ Validation failed. See details above.</p>'))


validate_button.on_click(on_validate_clicked)

display(
    instructions,
    widgets.VBox([widgets.HBox([upload_widget, data_type_dropdown, validate_button]), output]),
)
